# 📊 La Desigualdad en la Educación Financiera en República Dominicana
## Diagnóstico, Modelado de IA, Proyección de Costo-Efectividad y Estrategia Nacional
**Autor:** Proyecto Máster IA - Antigravity IDE (Google DeepMind)
**Fuentes:** Banco Central de la República Dominicana (BCRD), Superintendencia de Bancos (SB), MINERD, BID, OCDE, UNESCO, AFI.
---
### 🎯 Objetivo del Cuaderno
Analizar los datos empíricos sobre inclusión y alfabetización financiera en RD, construir modelos predictivos de Machine Learning (Scikit-Learn) para clasificar perfiles de vulnerabilidad económica, y proyectar la relación costo-efectividad del escalamiento nacional del programa educativo.

In [ ]:
# 1. Importación de Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Estilo de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'Arial'
print("Librerías cargadas exitosamente.")

## 1. Diagnóstico e Inclusión Financiera en RD (Datos ENIEF & BCRD)

In [ ]:
# Cargar datos del diagnóstico nacional
df_inclusion = pd.DataFrame({
    'Año': [2019, 2023],
    'Porcentaje_Producto_Financiero': [51, 55]
})

df_matricula = pd.DataFrame({
    'Nivel': ['Inicial', 'Primario', 'Secundario', 'Adultos'],
    'Estudiantes': [388743, 1186589, 861308, 203694]
})

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Inclusión financiera
sns.barplot(data=df_inclusion, x='Año', y='Porcentaje_Producto_Financiero', ax=ax[0], palette='Blues_d')
ax[0].set_title("Evolución de la Inclusión Financiera de Adultos en RD (%)", fontsize=12, fontweight='bold')
ax[0].set_ylim(0, 100)
for p in ax[0].patches:
    ax[0].annotate(f"{p.get_height():.0f}%", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                   ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# Gráfico 2: Matrícula escolar
sns.barplot(data=df_matricula, x='Nivel', y='Estudiantes', ax=ax[1], palette='crest')
ax[1].set_title("Matrícula del Sistema Educativo Dominicano (2024-2025)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Generación del Dataset Sintético Calibrado con la ENIEF

In [ ]:
# Generación de datos ajustados a las tasas de la ENIEF
np.random.seed(42)
n = 3000

zona = np.random.choice([0, 1], size=n, p=[0.35, 0.65]) # 0: Rural, 1: Urbano
genero = np.random.choice([0, 1], size=n, p=[0.52, 0.48]) # 0: Mujer, 1: Hombre
educacion = np.random.choice([0, 1, 2], size=n, p=[0.40, 0.45, 0.15]) # Primaria, Secundaria, Superior
ingreso = np.clip(np.random.lognormal(mean=9.8, sigma=0.7, size=n), 6000, 200000)
educ_finan = np.random.choice([0, 1], size=n, p=[0.92, 0.08]) # Recibió educación financiera

# Score logístico de inclusión
score = -1.2 + 0.6*zona + 0.3*genero + 0.8*educacion + 0.00003*ingreso + 1.5*educ_finan
prob = 1 / (1 + np.exp(-score))
bancarizado = (np.random.rand(n) < prob).astype(int)

# Perfil de Riesgo (0: Excluido/Alto Riesgo, 1: Vulnerable, 2: Sostenible)
perfil = np.where(bancarizado == 0, np.where(ingreso < 18000, 0, 1), np.where(educ_finan == 1, 2, 1))

df_sim = pd.DataFrame({
    'Zona': zona, 'Genero': genero, 'Educacion': educacion,
    'Ingreso_DOP': ingreso, 'Educacion_Financiera': educ_finan,
    'Bancarizado': bancarizado, 'Perfil_Riesgo': perfil
})

print("Vista preliminar del dataset:")
print(df_sim.head())

## 3. Entrenamiento del Modelo Predictivo (Random Forest Classifier)

In [ ]:
X = df_sim[['Zona', 'Genero', 'Educacion', 'Ingreso_DOP', 'Educacion_Financiera']]
y = df_sim['Perfil_Riesgo']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Modelo Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)

y_pred = rf.predict(X_scaled)
print("=== Reporte de Clasificación del Modelo IA ===")
print(classification_report(y, y_pred, target_names=['Alto Riesgo', 'Vulnerable', 'Sostenible']))

## 4. Agrupamiento K-Means para Segmentación Sociodemográfica

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
df_sim['Cluster'] = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_sim, x='Ingreso_DOP', y='Bancarizado', hue='Cluster', palette='Set1', alpha=0.6)
plt.xscale('log')
plt.title("Clusters de Inclusión Financiera según Ingreso (Escala Log)", fontsize=12, fontweight='bold')
plt.xlabel("Ingreso Mensual DOP (RD$)")
plt.show()

## 5. Proyección de Costo-Efectividad (Modelo Piloto BID Perú)

In [ ]:
costo_unitario_usd = 6.6
cobertura = {
    'Piloto (Fase 2)': 50000,
    'Secundaria Pública': 861308,
    'Primaria + Secundaria': 2047897
}

df_costos = pd.DataFrame([
    {'Fase': k, 'Estudiantes': v, 'Costo_Total_USD': v * costo_unitario_usd}
    for k, v in cobertura.items()
])

print("=== Proyección de Costo de Escalamiento Nacional ===")
for idx, row in df_costos.iterrows():
    print(f"-> {row['Fase']}: {row['Estudiantes']:,} estudiantes | Costo: US${row['Costo_Total_USD']:,.2f}")

## 6. Conclusiones y Recomendaciones de Política Pública
1. **Curricularización Obligatoria:** La evidencia internacional demuestra un impacto de **+0.14 DE** en estudiantes y **+0.30 DE** en docentes a un costo marginal de apenas **US$6.6 por estudiante**.
2. **Formación Docente vía INAFOCAM:** Garantizar la sostenibilidad capacitando a los formadores del sistema público.
3. **Evaluación de Impacto:** Implementar seguimiento continuo en alineación con las encuestas del Banco Central de la República Dominicana.